# 09b | A/B testing: sample sizes and analyzing results

Experiment take-homes and OAs usually hand you an experiment dataset and ask "is the new version better, and should it launch?" This page writes out all the computations you need.

About 55 minutes.

## 1. Sample size

Comparing two proportions, the sample size needed per group is:

$$n = \frac{(z_{1-\alpha/2} + z_{1-\beta})^2\,\big[p_1(1 - p_1) + p_2(1 - p_2)\big]}{(p_2 - p_1)^2}$$

where $p_1$ is the baseline conversion rate and $p_2 = p_1 + \mathrm{MDE}$.

### ✏️ Exercise 1 (Write it yourself)

Write a function `sample_size(p1, mde, alpha=0.05, power=0.8)` that returns the sample size needed per group from the formula above (no rounding needed). `mde` is the absolute lift.

Example: with a 10% baseline and a 1-percentage-point lift to detect, you need about 14750 users per group.

In [ ]:
from scipy import stats

def sample_size(p1, mde, alpha=0.05, power=0.8):
    # Write your code below
    pass

<details>
<summary><b>Hint</b></summary>

`z_alpha = stats.norm.ppf(1 - alpha / 2)`, `z_beta = stats.norm.ppf(power)`, then plug into the formula.

</details>

<details>
<summary><b>Answer</b></summary>

```python
from scipy import stats

def sample_size(p1, mde, alpha=0.05, power=0.8):
    p2 = p1 + mde
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power)
    return (z_alpha + z_beta) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)) / mde ** 2
```

Compare the first two tests: going from an MDE of 1% to 2% drops the sample size from about 14750 to about 3840, roughly a quarter. **Sample size is inversely proportional to the square of the MDE**, the rule you will use most often in interviews.

A ready-made function exists: `statsmodels.stats.power.NormalIndPower().solve_power(...)`, but interviews usually want you to write out the formula.

</details>

## 2. Analyzing an experiment dataset

Below is a simulated experiment dataset, one row per user: `group` (control / treatment), `converted` (whether the user converted), `revenue` (amount spent; 0 for most people).

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

rng = np.random.default_rng(42)
n_c, n_t = 5000, 5100
exp = pd.DataFrame({
    "user_id": np.arange(n_c + n_t),
    "group": ["control"] * n_c + ["treatment"] * n_t,
})
p = np.where(exp["group"] == "control", 0.100, 0.115)
exp["converted"] = (rng.random(len(exp)) < p).astype(int)
exp["revenue"] = exp["converted"] * rng.lognormal(3.0, 1.0, len(exp)).round(2)

exp.groupby("group").agg(n=("user_id", "count"), conv=("converted", "mean"), rev=("revenue", "mean"))

The **confidence interval for the difference** uses the **unpooled** standard error (each group's proportion on its own):

$$(\hat p_t - \hat p_c) \pm 1.96\sqrt{\frac{\hat p_t(1 - \hat p_t)}{n_t} + \frac{\hat p_c(1 - \hat p_c)}{n_c}}$$

**Testing H₀: the two groups are the same** uses the **pooled** standard error (as you wrote in 06b). The two are usually numerically very close, but knowing the difference earns points.

### ✏️ Exercise 2 (Write it yourself)

Using `exp`, compute:

- `diff`: treatment conversion rate minus control conversion rate;
- `ci_low`, `ci_high`: the 95% confidence interval for the difference (unpooled standard error, critical value 1.96);
- `pval`: the two-sided p-value of the two-proportion z-test (pooled standard error).

In [ ]:
c = exp[exp["group"] == "control"]["converted"]
t = exp[exp["group"] == "treatment"]["converted"]
# Write your code below

<details>
<summary><b>Answer</b></summary>

```python
c = exp[exp["group"] == "control"]["converted"]
t = exp[exp["group"] == "treatment"]["converted"]

p_c, p_t = c.mean(), t.mean()
n_c, n_t = len(c), len(t)
diff = p_t - p_c

se = np.sqrt(p_t * (1 - p_t) / n_t + p_c * (1 - p_c) / n_c)
ci_low, ci_high = diff - 1.96 * se, diff + 1.96 * se

pooled = (c.sum() + t.sum()) / (n_c + n_t)
se_pooled = np.sqrt(pooled * (1 - pooled) * (1 / n_c + 1 / n_t))
z = diff / se_pooled
pval = 2 * stats.norm.sf(abs(z))

print(round(diff, 4), (round(ci_low, 4), round(ci_high, 4)), round(pval, 4))
```

</details>

**🗣 Say it out loud:** Based on the results above, write a two- or three-sentence conclusion for a product manager.

<details>
<summary><b>Reference answer</b></summary>

"The treatment group's conversion rate is about 1.5 percentage points higher than control's (95% confidence interval roughly 0.3 to 2.7 percentage points), statistically significant at the 5% level. That is a relative lift of about 15%. The lower end of the interval is close to 0, so the true lift could be small; if launching is cheap and no guardrail metric got worse, I recommend launching; otherwise we could extend the experiment to narrow the interval."

Key points: lead with direction and size, then the uncertainty, then the recommendation; report both the absolute and the relative lift; don't just say "p < 0.05".

</details>

## 3. Heavy-tailed metrics like revenue

`revenue` is mostly 0, with a few large amounts. Compare means with Welch's t-test (fine in large samples thanks to the CLT), and double-check with a bootstrap.

### ✏️ Exercise 3 (Write it yourself)

Compare **revenue per user** between the groups:

- `rev_diff`: treatment revenue per user minus control revenue per user;
- `rev_p`: the p-value of Welch's t-test.

In [ ]:
rc = exp[exp["group"] == "control"]["revenue"]
rt = exp[exp["group"] == "treatment"]["revenue"]
# Write your code below

<details>
<summary><b>Answer</b></summary>

```python
rc = exp[exp["group"] == "control"]["revenue"]
rt = exp[exp["group"] == "treatment"]["revenue"]
rev_diff = rt.mean() - rc.mean()
rev_p = stats.ttest_ind(rt, rc, equal_var=False).pvalue
print(round(rev_diff, 3), round(rev_p, 4))
```

Conversion rate improved significantly, but revenue per user has a p-value of about 0.07 and is not significant. This is common: revenue has far more variance than conversion rate, so at the same sample size its power is much lower. It is also why primary metrics are often low-variance ones like conversion rate, or why revenue gets capped or adjusted with CUPED.

</details>

## 4. SRM check

The experiment planned a 50/50 split, but the actual counts are 5000 vs. 5100. Use a chi-square goodness-of-fit test: by default `stats.chisquare([observed counts per group])` expects equal counts in every group.

### ✏️ Exercise 4 (Write it yourself)

Write a function `srm_pvalue(n_a, n_b)`: given a planned 50/50 split, return the chi-square test p-value for observing `n_a` and `n_b`.

In [ ]:
from scipy import stats

def srm_pvalue(n_a, n_b):
    # Write your code below
    pass

print(srm_pvalue(5000, 5100))

<details>
<summary><b>Answer</b></summary>

```python
from scipy import stats

def srm_pvalue(n_a, n_b):
    return stats.chisquare([n_a, n_b]).pvalue

print(srm_pvalue(5000, 5100))
```

5000 vs. 5100 gives a p-value of about 0.32: no problem. 5000 vs. 5300 gives about 0.003, close to the common warning threshold of 0.001, and worth checking assignment and logging. The larger the sample, the more suspicious the same proportional imbalance becomes.

</details>

## 5. Simulating peeking

Below we simulate 2000 **A/A experiments** (no difference at all between the groups). Each experiment ends with 2000 users per group, and we check the p-value every 100 users, 20 times in total. Compare the false-positive rates of two approaches: look only once at the end; or look every time and declare significance as soon as any look is below 0.05.

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(0)
n_sim, n, step = 2000, 2000, 100
a = rng.normal(size=(n_sim, n))
b = rng.normal(size=(n_sim, n))             # same distribution as a: no true effect

final_p = stats.ttest_ind(a, b, axis=1).pvalue
ever_sig = np.zeros(n_sim, dtype=bool)
for k in range(step, n + 1, step):          # look every 100 users
    p_k = stats.ttest_ind(a[:, :k], b[:, :k], axis=1).pvalue
    ever_sig = ever_sig | (p_k < 0.05)

print("False-positive rate, one look at the end:", (final_p < 0.05).mean())
print("False-positive rate, 20 looks, stop when significant:", ever_sig.mean())

**🗣 Say it out loud:** Roughly what is the second number above? What does it tell you?

<details>
<summary><b>Reference answer</b></summary>

About 23%, four to five times the nominal 5% level. It shows that "check every day and stop once it's significant" makes the Type I error rate blow up: a change with no real effect has nearly a one-in-four chance of being declared "significant". The more often you look, the worse it gets. So either fix the sample size in advance or use a sequential method designed for repeated looks.

</details>

## Summary

- Sample size ∝ variance ÷ MDE²;
- Report: difference + confidence interval + p-value, with both absolute and relative lift;
- Use the unpooled standard error for the confidence interval and the pooled one for the test;
- Run the SRM check first, then look at the metrics;
- Heavy-tailed metrics have low power;
- Peeking multiplies the false-positive rate.

On to **10a**.